In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

print("library import is completed")

# 1. データの読み込み
path = "/kaggle/input/competitions/playground-series-s6e3/"
train_df = pd.read_csv(path + "train.csv")
test_df = pd.read_csv(path + "test.csv")

print("Data import is completed")

# ==========================================
# 特徴量エンジニアリング
# ==========================================
def create_features(df):
    df = df.copy()
    
    # 契約しているオプションサービスの数をカウント
    services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']
    # 'Yes'の数を数える
    df['Num_Services'] = df[services].apply(lambda x: (x == 'Yes').sum(), axis=1)
    
    # 料金と利用期間（tenure）に関する特徴量
    # 0割りを防ぐために + 1e-5
    df['Average_Monthly_Charges'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Charges_Difference'] = df['MonthlyCharges'] - df['Average_Monthly_Charges']
    
    return df

train_df = create_features(train_df)
test_df = create_features(test_df)

# ==========================================
# データの前処理
# ==========================================
# 目的変数(Churn)を 1/0 に変換
train_df["Churn"] = train_df["Churn"].map({'Yes': 1, 'No': 0})

# 特徴量(X)と目的変数(y)の分離 (不要な 'id' も削除)
X = train_df.drop(columns=["id", "Churn"])
y = train_df["Churn"]

X_test = test_df.drop(columns=["id", "Churn"], errors='ignore')

# カテゴリ変数の抽出と型変換
cat_features = X.select_dtypes(include=['object']).columns.tolist()
for col in cat_features:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# ==========================================
# 5-Fold Cross Validation による学習と予測
# ==========================================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# テストデータの予測値を格納する配列
test_preds = np.zeros(len(X_test))
# Out-Of-Fold（学習データに対する予測値）を格納する配列
oof_preds = np.zeros(len(X))

# LightGBMのハイパーパラメータ
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.03,
    'n_estimators': 2000,     # early_stoppingがあるので多めに設定
    'max_depth': 6,           # 深さを制限して過学習を防ぐ
    'num_leaves': 31,
    'colsample_bytree': 0.8,  # 特徴量をランダムにサンプリング
    'subsample': 0.8,         # データをランダムにサンプリング
    'random_state': 42,
    'n_jobs': -1
}

print("Start Training...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(200)
        ]
    )
    
    # 検証データへの予測
    val_pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred
    
    fold_auc = roc_auc_score(y_val, val_pred)
    print(f"Fold {fold+1} AUC: {fold_auc:.5f}")
    
    # テストデータへの予測（各Foldの予測の平均をとる）
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits

# 全体でのAUCスコアを確認
cv_auc = roc_auc_score(y, oof_preds)
print(f"\nOverall CV AUC: {cv_auc:.5f}")

# ==========================================
# 提出用ファイルの作成
# ==========================================
sub_df = pd.DataFrame({
    "id": test_df["id"],
    "Churn": test_preds
})

display(sub_df.head(10))

name = "submission_LGBM_CV_FE.csv"
sub_df.to_csv(name, index=False)
print(f"{name}の出力が完了しました")